In [1]:
from dsdlsupport.SplunkSearch import SplunkSearch
import pandas as pd, os, gc

SAVE_DIR = '/srv/work'

def load_and_save(search_obj, cloud, label):
    df = search_obj.as_df()
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['_time'] = pd.to_datetime(df['_time'], utc=True, errors='coerce')
    path = f'{SAVE_DIR}/te_{cloud}_raw_{label}.parquet'
    df.to_parquet(path, index=False, compression='snappy')
    mb_ram = df.memory_usage(deep=True).sum() / 1e6
    mb_disk = os.path.getsize(path) / 1e6
    print(f"[{cloud.upper()} {label}] {len(df):,} filas | "
          f"{str(df['_time'].min())[:19]} \u2192 {str(df['_time'].max())[:19]} | "
          f"RAM {mb_ram:.0f} MB \u2192 disco {mb_disk:.0f} MB")
    del df; gc.collect()
    return path


In [2]:
s_gcp_sep_w4 = SplunkSearch(search="""
index=thousandeyes sourcetype="cisco:thousandeyes:metric"
  earliest="09/25/2026:00:00:00" latest="10/01/2026:00:00:00"
| eval metric=case(
    isnotnull('metric_name:network.latency'),        "network.latency",
    isnotnull('metric_name:network.loss'),           "network.loss",
    isnotnull('metric_name:network.jitter'),         "network.jitter",
    isnotnull('metric_name:bgp.path_changes.count'), "bgp.path_changes.count",
    isnotnull('metric_name:bgp.reachability'),       "bgp.reachability",
    isnotnull('metric_name:bgp.updates.count'),      "bgp.updates.count"
)
| eval value=coalesce(
    'metric_name:network.latency','metric_name:network.loss','metric_name:network.jitter',
    'metric_name:bgp.path_changes.count','metric_name:bgp.reachability','metric_name:bgp.updates.count'
)
| where isnotnull(metric) AND cloud_provider="GCP"
| eval source_agent='thousandeyes.source.agent.name'
| eval target_agent='thousandeyes.target.agent.name'
| eval source_loc='thousandeyes.source.agent.location'
| eval target_loc='thousandeyes.target.agent.location'
| eval test_name='thousandeyes.test.name'
| eval test_type='thousandeyes.test.type'
| table _time source_agent target_agent source_loc target_loc
        cloud_provider region metric value test_name test_type
""")


In [3]:
load_and_save(s_gcp_sep_w4, 'gcp', 'sep_w4')
del s_gcp_sep_w4; gc.collect()


[GCP sep_w4] 5,971,560 filas | 2026-09-25 00:00:00 → 2026-09-30 23:59:58 | RAM 1545 MB → disco 38 MB


0

In [4]:
files = sorted(f for f in os.listdir('/srv/work') if f.endswith('.parquet'))
total_mb = 0
print(f"{'Archivo':<40} {'MB':>8}")
print('-' * 50)
for f in files:
    mb = os.path.getsize(f'/srv/work/{f}') / 1e6
    total_mb += mb
    print(f"{f:<40} {mb:>8.1f}")
print('-' * 50)
print(f"{'TOTAL':.<40} {total_mb:>8.1f} MB  ({len(files)} archivos)")


Archivo                                        MB
--------------------------------------------------
te_all_raw.parquet                          666.5
te_all_raw_1may_2jul.parquet               1323.8
te_all_raw_v3.parquet                      1735.0
te_aws_raw_apr_w1.parquet                    66.7
te_aws_raw_aug_w1.parquet                    57.4
te_aws_raw_aug_w2.parquet                    57.8
te_aws_raw_aug_w3.parquet                    58.8
te_aws_raw_aug_w4.parquet                    52.0
te_aws_raw_aug_w5.parquet                    38.8
te_aws_raw_jul_w1.parquet                    60.1
te_aws_raw_may_w1.parquet                    57.9
te_aws_raw_may_w2.parquet                    59.0
te_aws_raw_may_w3.parquet                    56.3
te_aws_raw_may_w4.parquet                    53.7
te_aws_raw_may_w5.parquet                    32.1
te_aws_raw_sep_w1.parquet                    55.1
te_aws_raw_sep_w2.parquet                    62.1
te_aws_raw_sep_w3.parquet                    59.8